# Local AI Sales Intelligence

A complete source walkthrough in ten chapters. Each chapter contains explanations, full file-writing code cells, runnable fictional-data examples and a verification checkpoint.

Start Jupyter from this project using its virtual environment. Read `README.md` first. The delivered source files are already present. Source cells reconstruct a separate copy in `notebook_project/`; demonstrations import the supplied, tested package.

All demonstrations use clearly fictional transactions. Model and n8n calls are opt-in. `Run All` does not install system software, download models, publish workflows, or start services. To review one chapter at a time, use the individual notebooks and pause at each checkpoint.

The original prompt's ten topics are retained. Basic deterministic preparation comes before optional local models, and the complete web API is assembled after its modules exist.


In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the supplied project or its notebooks directory."
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Source-writing cells reconstruct a separate copy and preserve the supplied project.
BUILD_ROOT = ROOT / "notebook_project"
BUILD_ROOT.mkdir(exist_ok=True)
for folder in ["sales_intelligence", "scripts", "tests", "workflows", "frontend/src", "docs"]:
    (BUILD_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_LOCAL_MODELS = False  # opt in only after installing local Ollama models
RUN_N8N = False          # opt in only after configuring/publishing the local workflows
print("Python:", sys.version.split()[0])
print("Source reconstruction:", BUILD_ROOT)


## Chapter 9 · Conversational API and background jobs

Expose authenticated FastAPI endpoints for uploading, preparation approvals, queries, exports and background ML/RAG jobs. A bounded process pool handles longer operations; clients poll task IDs and can request cancellation or one retry.

### Code cell · `sales_intelligence/jobs.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/jobs.py'}"
import multiprocessing
import os
import time
from concurrent.futures import ProcessPoolExecutor
from uuid import uuid4

import psutil

from .agents import run_agent
from .analytics import dashboard
from .config import Settings
from .ingestion import atomic_json
from .llm import LocalModel
from .ml import cluster, forecast, supervised
from .nlp import evaluate_text_classifier, text_analysis
from .rag import add_document
from .schemas import Chat, Filters, Forecast, Supervised, TextClassifier
from .store import Store, dumps, now


def set_status(store, task_id, status, progress, result=None, error=None):
    with store.db() as db:
        db.execute(
            "UPDATE tasks SET status=?,progress=?,result=?,error=?,updated=? WHERE id=?",
            (status, progress, dumps(result) if result is not None else None, error, now(), task_id),
        )


def execute_task(root, settings_values, task_id):
    # One worker and one model at a time by default; no external telemetry.
    os.environ["OMP_NUM_THREADS"] = "1"
    settings = Settings.model_validate(settings_values)
    store = Store(root)
    task = store.task(task_id)
    if task["cancel"]:
        set_status(store, task_id, "cancelled", 0)
        return
    payload = task["payload"]
    expected = payload["_dataset_version"]
    if store.get(task["dataset_id"])["version"] != expected:
        set_status(
            store, task_id, "failed", 0, error="Dataset changed before this task started. Submit again."
        )
        return
    set_status(store, task_id, "running", 10)
    started = time.monotonic()
    process = psutil.Process()
    cpu_start = process.cpu_times()
    try:
        kind, dataset_id = task["kind"], task["dataset_id"]
        request = payload["request"]
        if kind == "analyze":
            result = dashboard(store, dataset_id, Filters.model_validate(request))
        elif kind == "chat":
            result = run_agent(store, settings, dataset_id, Chat.model_validate(request), task_id)
        elif kind == "forecast":
            result = forecast(store, dataset_id, Forecast.model_validate(request))
        elif kind == "supervised":
            result = supervised(store, dataset_id, Supervised.model_validate(request))
        elif kind == "cluster":
            result = cluster(store, dataset_id)
        elif kind == "nlp":
            result = text_analysis(store, dataset_id)
        elif kind == "text_classifier":
            result = evaluate_text_classifier(store, dataset_id, TextClassifier.model_validate(request))
        elif kind == "document":
            path = store.directory(dataset_id) / request["staged_file"]
            try:
                result = add_document(
                    store, dataset_id, request["filename"], path.read_bytes(), LocalModel(settings, store)
                )
            finally:
                path.unlink(missing_ok=True)
        elif kind == "recommendations":
            result = run_agent(
                store, settings, dataset_id, Chat(question="خلاصه فروش", use_llm=settings.use_llm), task_id
            )
        else:
            raise ValueError("Unknown task kind.")
        if store.get(dataset_id)["version"] != expected:
            raise ValueError("Dataset changed during this task. Submit again on the current version.")
        if store.task(task_id)["cancel"]:
            set_status(store, task_id, "cancelled", 100)
            return
        if kind == "analyze":
            atomic_json(store.cache_path(dataset_id, "dashboard", request, version=expected), result)
        cpu_end = process.cpu_times()
        metrics = {
            "wall_seconds": time.monotonic() - started,
            "cpu_seconds": cpu_end.user + cpu_end.system - cpu_start.user - cpu_start.system,
            "resident_ram_bytes_at_completion": process.memory_info().rss,
        }
        set_status(store, task_id, "completed", 100, {"output": result, "metrics": metrics})
        store.audit(dataset_id, "task_completed", {"task_id": task_id, "kind": kind, "metrics": metrics})
    except Exception as exc:  # noqa: BLE001 - worker exceptions must become a persisted terminal task status
        # Do not expose model prompts, raw records, credentials or parser internals in API errors.
        message = (
            str(exc)[:500]
            if isinstance(exc, ValueError)
            else f"{type(exc).__name__}: task failed; inspect local configuration."
        )
        set_status(store, task_id, "failed", 100, error=message)


class JobQueue:
    def __init__(self, store, settings):
        self.store, self.settings = store, settings
        self.pool = ProcessPoolExecutor(
            max_workers=settings.workers, mp_context=multiprocessing.get_context("spawn")
        )

    def enqueue(self, dataset_id, kind, request):
        data = self.store.get(dataset_id)
        if "cleaned" not in data["meta"]:
            raise ValueError("Confirm mapping and cleaning before submitting analysis jobs.")
        payload = dumps({"request": request, "_dataset_version": data["version"]})
        with self.store.db() as db:
            db.execute("BEGIN IMMEDIATE")
            existing = db.execute(
                "SELECT id,status FROM tasks WHERE dataset_id=? AND kind=? AND payload=?",
                (dataset_id, kind, payload),
            ).fetchone()
            if existing:
                return self.store.task(existing["id"])
            count = db.execute("SELECT COUNT(*) FROM tasks WHERE status IN ('queued','running')").fetchone()[
                0
            ]
            if count >= 16:
                raise ValueError("Local task queue is full. Wait for an existing task to finish.")
            task_id = str(uuid4())
            db.execute(
                "INSERT INTO tasks(id,dataset_id,kind,status,created,updated,payload) VALUES (?,?,?,'queued',?,?,?)",
                (task_id, dataset_id, kind, now(), now(), payload),
            )
        self.pool.submit(execute_task, str(self.store.root), self.settings.model_dump(mode="json"), task_id)
        return self.store.task(task_id)

    def retry(self, task_id):
        task = self.store.task(task_id)
        if task["status"] not in {"failed", "cancelled"} or task["kind"] == "document":
            raise ValueError("Only failed/cancelled analysis tasks can be retried. Upload documents again.")
        if self.store.get(task["dataset_id"])["version"] != task["payload"]["_dataset_version"]:
            raise ValueError("Dataset changed; create a new request.")
        # At most one explicit retry per task, tracked by an audit event.
        with self.store.db() as db:
            db.execute("BEGIN IMMEDIATE")
            prior = db.execute(
                "SELECT COUNT(*) FROM audit WHERE action='task_retry' AND detail=?",
                (dumps({"task_id": task_id}),),
            ).fetchone()[0]
            if prior:
                raise ValueError("Retry limit reached. Correct the cause and change the request.")
            db.execute(
                "UPDATE tasks SET status='queued',cancel=0,error=NULL,progress=0 WHERE id=?", (task_id,)
            )
            db.execute(
                "INSERT INTO audit(dataset_id,created,action,detail) VALUES (?,?,?,?)",
                (task["dataset_id"], now(), "task_retry", dumps({"task_id": task_id})),
            )
        self.pool.submit(execute_task, str(self.store.root), self.settings.model_dump(mode="json"), task_id)
        return self.store.task(task_id)

    def close(self):
        self.pool.shutdown(wait=True, cancel_futures=False)


### Code cell · `sales_intelligence/api.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/api.py'}"
import hmac
import json
from contextlib import asynccontextmanager
from typing import Annotated
from uuid import UUID, uuid4

from fastapi import APIRouter, Depends, FastAPI, File, Header, HTTPException, Query, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response
from starlette.concurrency import run_in_threadpool

from .config import Settings
from .ingestion import ingest
from .jobs import JobQueue
from .preparation import clean, inspect, load_clean, save_mapping, save_relationship
from .reporting import export, simulate
from .schemas import (
    Chat,
    Cleaning,
    Filters,
    Forecast,
    Mapping,
    Relationship,
    Simulation,
    Supervised,
    TextClassifier,
)
from .store import Store, now


def create_app(settings=None):
    settings = (settings or Settings()).prepare()
    store = Store(settings.data_dir)
    queue = None

    @asynccontextmanager
    async def lifespan(app):
        nonlocal queue
        with store.db() as db:
            db.execute(
                "UPDATE tasks SET status='failed',error='API restarted. Retry explicitly.',updated=? WHERE status IN ('queued','running')",
                (now(),),
            )
        queue = JobQueue(store, settings)
        yield
        queue.close()

    app = FastAPI(
        title="Local AI Sales Intelligence",
        version="0.1.0",
        lifespan=lifespan,
        docs_url=None,
        redoc_url=None,
        openapi_url=None,
    )
    app.state.store = store
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
        allow_methods=["GET", "POST"],
        allow_headers=["X-API-Key", "Content-Type"],
    )

    def authorize(x_api_key: Annotated[str | None, Header()] = None):
        if x_api_key is None or not hmac.compare_digest(x_api_key, settings.api_key):
            raise HTTPException(401, "A valid X-API-Key is required.")

    router = APIRouter(prefix="/api", dependencies=[Depends(authorize)])

    @app.exception_handler(ValueError)
    async def invalid(request, exc):
        return JSONResponse(status_code=400, content={"detail": str(exc)[:1000]})

    @app.exception_handler(KeyError)
    async def missing(request, exc):
        return JSONResponse(status_code=404, content={"detail": "Dataset or task not found."})

    @router.get("/health")
    def health():
        return {"status": "ok", "local_llm_enabled": settings.use_llm, "single_company": True}

    @router.post("/upload", status_code=201)
    async def upload(file: Annotated[UploadFile, File()]):
        try:
            body = await file.read(settings.max_upload_mb * 1024 * 1024 + 1)
            return await run_in_threadpool(ingest, store, settings, file.filename or "upload", body)
        finally:
            await file.close()

    @router.get("/jobs/{job_id}")
    def dataset(job_id: UUID):
        return store.get(str(job_id))

    @router.get("/sheets/{job_id}")
    def sheets(job_id: UUID):
        return store.get(str(job_id))["meta"]["sheets"]

    @router.get("/schema/{job_id}")
    def schema(job_id: UUID):
        return inspect(store, str(job_id))

    @router.post("/mapping/{job_id}")
    def mapping(job_id: UUID, request: Mapping):
        return save_mapping(store, str(job_id), request)

    @router.post("/relationships/{job_id}")
    def relationships(job_id: UUID, request: Relationship):
        return save_relationship(store, str(job_id), request)

    @router.post("/clean/{job_id}")
    def cleaning(job_id: UUID, request: Cleaning):
        return clean(store, str(job_id), request)

    def enqueue(job_id, kind, request):
        if queue is None:
            raise HTTPException(503, "Start the API with its lifespan enabled.")
        task = queue.enqueue(str(job_id), kind, request)
        return {
            "task_id": task["id"],
            "dataset_id": str(job_id),
            "status": task["status"],
            "status_url": f"/api/tasks/{task['id']}",
        }

    @router.post("/analyze/{job_id}", status_code=202)
    def analysis(job_id: UUID, filters: Filters):
        return enqueue(job_id, "analyze", filters.model_dump())

    @router.get("/dashboard/{job_id}")
    def get_dashboard(job_id: UUID, filters: Annotated[Filters, Query()]):
        # Return a completed dashboard artifact; CPU-heavy generation is submitted with /analyze.
        path = store.cache_path(str(job_id), "dashboard", filters.model_dump())
        if path.exists():
            return json.loads(path.read_text())
        raise HTTPException(409, "Submit POST /api/analyze/{dataset_id} and retrieve the task output.")

    @router.post("/chat/{job_id}", status_code=202)
    def chat(job_id: UUID, request: Chat):
        return enqueue(job_id, "chat", request.model_dump())

    @router.get("/recommendations/{job_id}", status_code=202)
    def recommendations(job_id: UUID):
        return enqueue(job_id, "recommendations", {})

    @router.post("/ml/forecast/{job_id}", status_code=202)
    def predict(job_id: UUID, request: Forecast):
        return enqueue(job_id, "forecast", request.model_dump())

    @router.post("/ml/supervised/{job_id}", status_code=202)
    def train(job_id: UUID, request: Supervised):
        return enqueue(job_id, "supervised", request.model_dump())

    @router.post("/ml/cluster/{job_id}", status_code=202)
    def clustering(job_id: UUID):
        return enqueue(job_id, "cluster", {})

    @router.post("/nlp/{job_id}", status_code=202)
    def nlp(job_id: UUID):
        return enqueue(job_id, "nlp", {})

    @router.post("/nlp/classifier/{job_id}", status_code=202)
    def text_classifier(job_id: UUID, request: TextClassifier):
        return enqueue(job_id, "text_classifier", request.model_dump())

    @router.post("/rag/documents/{job_id}", status_code=202)
    async def document(job_id: UUID, file: Annotated[UploadFile, File()]):
        if not settings.use_llm:
            raise HTTPException(409, "Enable local models and download the embedding model first.")
        directory = store.directory(str(job_id))
        staged = f"document_upload_{uuid4().hex}.tmp"
        path = directory / staged
        try:
            body = await file.read(10 * 1024 * 1024 + 1)
            if len(body) > 10 * 1024 * 1024 or not body:
                raise ValueError("Document exceeds the 10 MB limit or is empty.")
            path.write_bytes(body)
            return enqueue(
                job_id, "document", {"staged_file": staged, "filename": file.filename or "document.txt"}
            )
        except Exception:
            path.unlink(missing_ok=True)
            raise
        finally:
            await file.close()

    @router.post("/simulation/{job_id}")
    def scenario(job_id: UUID, request: Simulation):
        return simulate(store, str(job_id), request)

    @router.get("/export/{job_id}")
    def download(job_id: UUID, format: str = "json"):
        content, mime, name = export(store, str(job_id), format)
        return Response(
            content, media_type=mime, headers={"Content-Disposition": f'attachment; filename="{name}"'}
        )

    @router.get("/preview/{job_id}")
    def preview(job_id: UUID, offset: int = Query(0, ge=0), limit: int = Query(25, ge=1, le=100)):
        frame, _ = load_clean(store, str(job_id))
        return {
            "total": len(frame),
            "offset": offset,
            "rows": json.loads(
                frame.iloc[offset : offset + limit].to_json(orient="records", date_format="iso")
            ),
        }

    @router.get("/tasks/{task_id}")
    def task(task_id: UUID):
        task = store.task(str(task_id))
        task.pop("payload", None)
        return task

    @router.post("/tasks/{task_id}/cancel")
    def cancel(task_id: UUID):
        task = store.task(str(task_id))
        if task["status"] in {"completed", "failed", "cancelled"}:
            return {"status": task["status"]}
        with store.db() as db:
            db.execute("UPDATE tasks SET cancel=1,updated=? WHERE id=?", (now(), str(task_id)))
        return {
            "status": "cancellation_requested",
            "note": "Cooperative cancellation takes effect at a task boundary.",
        }

    @router.post("/tasks/{task_id}/retry", status_code=202)
    def retry(task_id: UUID):
        if queue is None:
            raise HTTPException(503, "Job queue is unavailable.")
        return queue.retry(str(task_id))

    @router.get("/audit/{job_id}")
    def audit(job_id: UUID, offset: int = Query(0, ge=0), limit: int = Query(50, ge=1, le=100)):
        store.get(str(job_id))
        with store.db() as db:
            rows = db.execute(
                "SELECT * FROM audit WHERE dataset_id=? ORDER BY id LIMIT ? OFFSET ?",
                (str(job_id), limit, offset),
            ).fetchall()
        return [{**dict(row), "detail": json.loads(row["detail"])} for row in rows]

    app.include_router(router)
    return app


### Code cell · `scripts/serve.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/serve.py'}"
import os

import uvicorn

if __name__ == "__main__":
    uvicorn.run(
        "sales_intelligence.api:create_app",
        factory=True,
        host=os.environ.get("SALES_BIND", "127.0.0.1"),
        port=8000,
        workers=1,
    )


### Run the chapter example

In [ ]:
if "demo_store" not in globals():
    from scripts.notebook_demo import make_demo
    demo_store, demo_settings, demo_id, sample_frame = make_demo(ROOT)
print("Fictional dataset:", demo_id)
from fastapi.testclient import TestClient
from sales_intelligence.api import create_app
# This in-process check starts no public server and uses only fictional credentials.
app = create_app(demo_settings)
client = TestClient(app)
assert client.get("/api/health").status_code == 401
client.headers["X-API-Key"] = demo_settings.api_key
assert client.get("/api/health").status_code == 200
display(client.get("/api/schema/" + demo_id).json())


### Review checkpoint

Run one API server worker; its internal process pool is configurable. Cancellation is cooperative at task boundaries. A restart marks incomplete jobs failed for explicit retry. Dataset changes invalidate queued and in-flight results.

Review this result before moving to the next chapter.